# Day 03 - Part 1: Pythonic Thinking
### Python → Generative AI Engineer Journey

---
**Author:** Shaurab Kumar Jha  
**Date:** Day 3 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

| # | Topic |
|---|-------|
| 1 | Iterators & Iterables - protocol deep dive |
| 2 | Generators - `yield`, `yield from`, generator expressions |
| 3 | `zip`, `enumerate`, `map`, `filter`, `reduce` |
| 4 | `*args` and `**kwargs` - complete patterns |
| 5 | Shallow vs Deep Copy - the real difference |
| 6 | Bonus: `any`, `all`, `sorted`, `min/max` with key |

Every concept is covered to the bottom. Nothing skipped.

---
# PART 1 - ITERATORS & ITERABLES

## Theory

These two words are often confused. They are different things.

| Term | Definition | Has `__iter__`? | Has `__next__`? |
|------|-----------|----------------|----------------|
| **Iterable** | Anything you can loop over | ✅ | ❌ (usually) |
| **Iterator** | Object that produces values one at a time | ✅ | ✅ |

**The Iterator Protocol:**
1. `__iter__()` → returns the iterator object itself
2. `__next__()` → returns next value, raises `StopIteration` when done

A `for` loop is **syntactic sugar** for:
```python
it = iter(obj)          # calls obj.__iter__()
while True:
    try:
        val = next(it)  # calls it.__next__()
    except StopIteration:
        break
```

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.1  UNDER THE HOOD - what for loop actually does
# ═══════════════════════════════════════════════════════════════════════════════

my_list = [10, 20, 30]

# What Python does internally when you write: for x in my_list
iterator = iter(my_list)          # Step 1: get iterator
print(type(iterator))             # <class 'list_iterator'>

print(next(iterator))             # Step 2: get first value → 10
print(next(iterator))             # → 20
print(next(iterator))             # → 30
try:
    print(next(iterator))         # → StopIteration!
except StopIteration:
    print("StopIteration raised - loop ends here")

# Iterables you already know:
for obj in ["list", (1,2), {"a":1}, {1,2}, "string", range(3)]:
    it = iter(obj)
    print(f"{str(type(obj).__name__):<12} → iterator type: {type(it).__name__}")

<class 'list_iterator'>
10
20
30
StopIteration raised - loop ends here
str          → iterator type: str_ascii_iterator
tuple        → iterator type: tuple_iterator
dict         → iterator type: dict_keyiterator
set          → iterator type: set_iterator
str          → iterator type: str_ascii_iterator
range        → iterator type: range_iterator


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.2  BUILD YOUR OWN ITERATOR CLASS
# ═══════════════════════════════════════════════════════════════════════════════

class CountDown:
    """
    Custom iterator that counts down from n to 1.
    Implements the iterator protocol: __iter__ + __next__.
    """

    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        """An iterator must return itself from __iter__."""
        return self

    def __next__(self):
        """Return next value or raise StopIteration."""
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val


cd = CountDown(5)
print("CountDown(5):", list(cd))    # [5, 4, 3, 2, 1]

# Works in for loop naturally
print("In for loop:", end=" ")
for n in CountDown(3):
    print(n, end=" ")
print()

# Works with all iterator tools
print("sum        :", sum(CountDown(5)))
print("max        :", max(CountDown(5)))
print("list       :", list(CountDown(5)))


# ── Iterable vs Iterator distinction ─────────────────────────────────────────
class NumberRange:
    """
    Iterable (NOT iterator) - returns a fresh iterator every time.
    This means you can loop over it multiple times.
    """
    def __init__(self, start, stop):
        self.start = start
        self.stop  = stop

    def __iter__(self):
        """Returns a new iterator each time - key difference."""
        return CountDown(self.stop - self.start)


r = NumberRange(1, 4)
print("Loop 1:", list(r))    # [3, 2, 1]
print("Loop 2:", list(r))    # [3, 2, 1] - works again! (iterable, not iterator)

CountDown(5): [5, 4, 3, 2, 1]
In for loop: 3 2 1 
sum        : 15
max        : 5
list       : [5, 4, 3, 2, 1]
Loop 1: [3, 2, 1]
Loop 2: [3, 2, 1]


---
# PART 2 - GENERATORS

## Theory

A **generator** is a function that uses `yield` instead of `return`.  
It is automatically an iterator - Python builds `__iter__` and `__next__` for you.

**Why generators?**  
- **Memory efficient** - values produced one at a time, not all at once in RAM.
- `range(1_000_000)` uses ~48 bytes. `list(range(1_000_000))` uses ~8 MB.
- Essential for streaming data, pipelines, infinite sequences.

**`yield` vs `return`:**
| | `return` | `yield` |
|-|----------|---------|
| Function type | Regular | Generator |
| Resumes? | ❌ - terminates | ✅ - pauses & resumes |
| Values | One | Many (lazy) |

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.1  BASIC GENERATOR FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def count_up(start: int, stop: int):
    """
    Generator that yields integers from start to stop-1.
    Execution pauses at each yield and resumes on next().
    """
    print("  [generator] Starting...")
    current = start
    while current < stop:
        print(f"  [generator] About to yield {current}")
        yield current              # ← pause here, return value
        print(f"  [generator] Resumed after yielding {current}")
        current += 1
    print("  [generator] Done.")


gen = count_up(1, 4)
print(f"type: {type(gen)}")        # <class 'generator'>
print(f"is iterator: {hasattr(gen, '__next__')}")
print()
print("Calling next() 3 times:")
print(f"  next() → {next(gen)}")
print(f"  next() → {next(gen)}")
print(f"  next() → {next(gen)}")
try:
    next(gen)                      # Generator exhausted
except StopIteration:
    print("  StopIteration raised")

type: <class 'generator'>
is iterator: True

Calling next() 3 times:
  [generator] Starting...
  [generator] About to yield 1
  next() → 1
  [generator] Resumed after yielding 1
  [generator] About to yield 2
  next() → 2
  [generator] Resumed after yielding 2
  [generator] About to yield 3
  next() → 3
  [generator] Resumed after yielding 3
  [generator] Done.
  StopIteration raised


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.2  MEMORY COMPARISON - list vs generator
# ═══════════════════════════════════════════════════════════════════════════════

import sys

N = 1_000_000

list_version = [x * 2 for x in range(N)]          # List comprehension
gen_version  = (x * 2 for x in range(N))           # Generator expression

print(f"List  size: {sys.getsizeof(list_version):>10,} bytes  ({sys.getsizeof(list_version)/1024/1024:.1f} MB)")
print(f"Gen   size: {sys.getsizeof(gen_version):>10,} bytes  ({sys.getsizeof(gen_version)} bytes - constant!)")

print()
print("Generator expression syntax: (expr for item in iterable if cond)")
print("List comprehension syntax  : [expr for item in iterable if cond]")

# Both produce same results but generator is lazy
gen_sum  = sum(x * 2 for x in range(N))            # Generator used directly
list_sum = sum([x * 2 for x in range(N)])
print(f"\nSums equal? {gen_sum == list_sum}")

List  size:  8,448,728 bytes  (8.1 MB)
Gen   size:        200 bytes  (200 bytes - constant!)

Generator expression syntax: (expr for item in iterable if cond)
List comprehension syntax  : [expr for item in iterable if cond]

Sums equal? True


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.3  PRACTICAL GENERATORS
# ═══════════════════════════════════════════════════════════════════════════════

# ── Infinite sequence generator ───────────────────────────────────────────────
def fibonacci():
    """
    Infinite Fibonacci sequence generator.
    Never terminates - caller decides how many to take.
    """
    a, b = 0, 1
    while True:             # Infinite loop - OK in generators
        yield a
        a, b = b, a + b


# Take first 15 Fibonacci numbers
import itertools
fib_gen   = fibonacci()
first_15  = [next(fib_gen) for _ in range(15)]
print(f"First 15 Fibonacci: {first_15}")

# ── File-reading generator (memory efficient for huge files) ──────────────────
def read_in_chunks(text: str, chunk_size: int = 20):
    """Yield text in fixed-size chunks - useful for large files."""
    for i in range(0, len(text), chunk_size):
        yield text[i:i + chunk_size]


sample = "The quick brown fox jumps over the lazy dog. Python is amazing!"
print("\nText in chunks of 20:")
for i, chunk in enumerate(read_in_chunks(sample, 20), 1):
    print(f"  Chunk {i}: {chunk!r}")

# ── Pipeline with generators ──────────────────────────────────────────────────
def integers(n):       yield from range(n)
def squared(seq):      yield from (x**2 for x in seq)
def only_even(seq):    yield from (x for x in seq if x % 2 == 0)

# Chain them: integers → squared → filter even
pipeline = only_even(squared(integers(10)))
print(f"\nPipeline (0–9 → squared → even): {list(pipeline)}")

First 15 Fibonacci: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]

Text in chunks of 20:
  Chunk 1: 'The quick brown fox '
  Chunk 2: 'jumps over the lazy '
  Chunk 3: 'dog. Python is amazi'
  Chunk 4: 'ng!'

Pipeline (0–9 → squared → even): [0, 4, 16, 36, 64]


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.4  yield from  - delegating to sub-generators
# ═══════════════════════════════════════════════════════════════════════════════

def flatten_gen(nested):
    """
    Recursively flatten nested iterables using yield from.
    yield from delegates to another iterable/generator.
    """
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten_gen(item)   # Recursive delegation
        else:
            yield item


data = [1, [2, 3], [4, [5, 6]], [7, [8, [9]]]]
print(f"Nested  : {data}")
print(f"Flat    : {list(flatten_gen(data))}")

# yield from with multiple sources
def chain_sources(*iterables):
    """Yield from each iterable in sequence (like itertools.chain)."""
    for iterable in iterables:
        yield from iterable


result = list(chain_sources([1, 2], (3, 4), {5, 6}, "ab"))
print(f"Chained : {result}")

Nested  : [1, [2, 3], [4, [5, 6]], [7, [8, [9]]]]
Flat    : [1, 2, 3, 4, 5, 6, 7, 8, 9]
Chained : [1, 2, 3, 4, 5, 6, 'a', 'b']


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.5  GENERATOR METHODS - send(), throw(), close()
# ═══════════════════════════════════════════════════════════════════════════════

def accumulator():
    """
    Two-way generator: receives values via send().
    yield acts as both input and output.
    """
    total = 0
    while True:
        value = yield total    # yield sends total OUT, receives value IN
        if value is None:
            break
        total += value


acc = accumulator()
next(acc)            # Prime the generator (advance to first yield)
print(f"send(10): {acc.send(10)}")
print(f"send(20): {acc.send(20)}")
print(f"send(30): {acc.send(30)}")

# close() - throws GeneratorExit inside the generator
acc.close()
print("Generator closed.")

send(10): 10
send(20): 30
send(30): 60
Generator closed.


---
# PART 3 - BUILT-IN FUNCTIONAL TOOLS
## `zip`, `enumerate`, `map`, `filter`, `reduce`

All of these are **lazy** - they return iterators, not lists.  
Wrap with `list()` to see the values.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.1  zip() - pair up elements from multiple iterables
# ═══════════════════════════════════════════════════════════════════════════════

names   = ["Alice", "Bob", "Carol", "Dave"]
scores  = [92, 85, 78, 91]
cities  = ["Delhi", "Mumbai", "Pune"]

# Basic zip - stops at the shortest iterable
paired = list(zip(names, scores))
print(f"zip(names, scores)    : {paired}")

# zip with 3 iterables
tripled = list(zip(names, scores, cities))
print(f"zip(3 iterables)      : {tripled}")

# zip_longest - fills missing values with fillvalue
from itertools import zip_longest
full = list(zip_longest(names, scores, cities, fillvalue="N/A"))
print(f"zip_longest           : {full}")

# Unzipping (unpacking a zipped list)
zipped = [("a", 1), ("b", 2), ("c", 3)]
keys, vals = zip(*zipped)          # * unpacks the list of tuples
print(f"\nUnzip keys  : {keys}")
print(f"Unzip vals  : {vals}")

# Real use: build dict from two parallel lists
subjects    = ["Math", "Science", "CS"]
pass_marks  = [40, 40, 50]
cutoffs = dict(zip(subjects, pass_marks))
print(f"\nDict from zip: {cutoffs}")

# zip for parallel iteration with index
print("\nLeaderboard:")
for rank, (name, score) in enumerate(zip(names, scores), 1):
    print(f"  #{rank}  {name:<8}  {score}")

zip(names, scores)    : [('Alice', 92), ('Bob', 85), ('Carol', 78), ('Dave', 91)]
zip(3 iterables)      : [('Alice', 92, 'Delhi'), ('Bob', 85, 'Mumbai'), ('Carol', 78, 'Pune')]
zip_longest           : [('Alice', 92, 'Delhi'), ('Bob', 85, 'Mumbai'), ('Carol', 78, 'Pune'), ('Dave', 91, 'N/A')]

Unzip keys  : ('a', 'b', 'c')
Unzip vals  : (1, 2, 3)

Dict from zip: {'Math': 40, 'Science': 40, 'CS': 50}

Leaderboard:
  #1  Alice     92
  #2  Bob       85
  #3  Carol     78
  #4  Dave      91


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.2  enumerate() - index + value together
# ═══════════════════════════════════════════════════════════════════════════════

fruits = ["apple", "banana", "cherry", "mango"]

# Without enumerate (anti-pattern in Python)
for i in range(len(fruits)):
    print(f"  [non-pythonic] {i}: {fruits[i]}")

print()

# With enumerate (Pythonic)
for i, fruit in enumerate(fruits):
    print(f"  [pythonic]     {i}: {fruit}")

# Custom start index
print("\nNumbered list (start=1):")
for i, fruit in enumerate(fruits, start=1):
    print(f"  {i}. {fruit}")

# enumerate returns (index, value) tuples
indexed = list(enumerate(fruits))
print(f"\nlist(enumerate(fruits)) = {indexed}")

# Use case: find index of specific items
target = "cherry"
for i, fruit in enumerate(fruits):
    if fruit == target:
        print(f"Found '{target}' at index {i}")

# Use case: modify list while tracking index
scores = [85, 40, 92, 38, 76]
results = [(i, s, "PASS" if s >= 50 else "FAIL") for i, s in enumerate(scores)]
print(f"\nScores with result: {results}")

  [non-pythonic] 0: apple
  [non-pythonic] 1: banana
  [non-pythonic] 2: cherry
  [non-pythonic] 3: mango

  [pythonic]     0: apple
  [pythonic]     1: banana
  [pythonic]     2: cherry
  [pythonic]     3: mango

Numbered list (start=1):
  1. apple
  2. banana
  3. cherry
  4. mango

list(enumerate(fruits)) = [(0, 'apple'), (1, 'banana'), (2, 'cherry'), (3, 'mango')]
Found 'cherry' at index 2

Scores with result: [(0, 85, 'PASS'), (1, 40, 'FAIL'), (2, 92, 'PASS'), (3, 38, 'FAIL'), (4, 76, 'PASS')]


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.3  map() - apply function to every element
# ═══════════════════════════════════════════════════════════════════════════════

# Syntax: map(function, iterable)  - returns a MAP object (lazy iterator)

numbers = [1, 2, 3, 4, 5]

# map with built-in function
squared = list(map(lambda x: x**2, numbers))
print(f"Squared       : {squared}")

# map with named function
def celsius_to_fahrenheit(c):
    return round(c * 9/5 + 32, 1)

temps_c = [0, 20, 37, 100]
temps_f = list(map(celsius_to_fahrenheit, temps_c))
print(f"Celsius   : {temps_c}")
print(f"Fahrenheit: {temps_f}")

# map with multiple iterables
a = [1, 2, 3]
b = [10, 20, 30]
sums = list(map(lambda x, y: x + y, a, b))
print(f"\nElement-wise sum: {sums}")

# map with type conversion
str_nums = ["1", "2", "3", "4", "5"]
int_nums = list(map(int, str_nums))     # Most common use
print(f"str→int: {int_nums}")

# Pythonic: list comprehension vs map
# Both equivalent - comprehension often more readable
via_map   = list(map(lambda x: x**2, range(5)))
via_comp  = [x**2 for x in range(5)]
print(f"\nmap result   : {via_map}")
print(f"comp result  : {via_comp}")
print(f"Equal        : {via_map == via_comp}")

Squared       : [1, 4, 9, 16, 25]
Celsius   : [0, 20, 37, 100]
Fahrenheit: [32.0, 68.0, 98.6, 212.0]

Element-wise sum: [11, 22, 33]
str→int: [1, 2, 3, 4, 5]

map result   : [0, 1, 4, 9, 16]
comp result  : [0, 1, 4, 9, 16]
Equal        : True


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.4  filter() - keep only elements matching condition
# ═══════════════════════════════════════════════════════════════════════════════

# Syntax: filter(function, iterable)
# function must return True/False
# If function is None, filters out falsy values

numbers = [1, -2, 3, -4, 5, 0, -6, 7]

positives = list(filter(lambda x: x > 0, numbers))
print(f"Positives     : {positives}")

# filter with None removes falsy values
messy = [0, 1, "", "hello", None, False, True, [], [1, 2]]
truthy_only = list(filter(None, messy))
print(f"Truthy only   : {truthy_only}")

# filter with named function
def is_prime(n):
    """Return True if n is a prime number."""
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

primes = list(filter(is_prime, range(50)))
print(f"Primes < 50   : {primes}")

# filter on list of dicts (real-world use)
employees = [
    {"name": "Alice", "dept": "Engineering", "salary": 120000},
    {"name": "Bob",   "dept": "Marketing",   "salary": 85000},
    {"name": "Carol", "dept": "Engineering", "salary": 110000},
    {"name": "Dave",  "dept": "HR",          "salary": 75000},
]
engineers = list(filter(lambda e: e["dept"] == "Engineering", employees))
print(f"\nEngineers: {[e['name'] for e in engineers]}")

# Equivalent comprehension
engineers_comp = [e for e in employees if e["dept"] == "Engineering"]
print(f"Same via comp: {[e['name'] for e in engineers_comp]}")

Positives     : [1, 3, 5, 7]
Truthy only   : [1, 'hello', True, [1, 2]]
Primes < 50   : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

Engineers: ['Alice', 'Carol']
Same via comp: ['Alice', 'Carol']


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.5  reduce() - fold iterable into single value
# ═══════════════════════════════════════════════════════════════════════════════

from functools import reduce

# reduce(function, iterable, initial_value)
# Applies function cumulatively:
#   reduce(f, [a, b, c, d]) → f(f(f(a, b), c), d)

numbers = [1, 2, 3, 4, 5]

# Sum (equivalent to built-in sum())
total = reduce(lambda acc, x: acc + x, numbers)
print(f"reduce sum     : {total}")

# Product
product = reduce(lambda acc, x: acc * x, numbers)
print(f"reduce product : {product}")

# Max (equivalent to max())
maximum = reduce(lambda a, b: a if a > b else b, numbers)
print(f"reduce max     : {maximum}")

# With initial value
total_from_100 = reduce(lambda acc, x: acc + x, numbers, 100)
print(f"reduce sum+100 : {total_from_100}")

# Real-world: flatten nested list
nested = [[1, 2], [3, 4], [5, 6]]
flat   = reduce(lambda acc, lst: acc + lst, nested)
print(f"\nreduce flatten : {flat}")

# Building a pipeline function with reduce
def compose(*funcs):
    """Compose multiple functions: compose(f, g, h)(x) → f(g(h(x)))."""
    return reduce(lambda f, g: lambda x: f(g(x)), funcs)

double   = lambda x: x * 2
add_ten  = lambda x: x + 10
square   = lambda x: x ** 2

pipeline = compose(double, add_ten, square)   # double(add_ten(square(x)))
print(f"\ncompose(double, add_ten, square)(3) = {pipeline(3)}")
print(f"  Step by step: square(3)={3**2}, add_ten({3**2})={3**2+10}, double({3**2+10})={(3**2+10)*2}")

reduce sum     : 15
reduce product : 120
reduce max     : 5
reduce sum+100 : 115

reduce flatten : [1, 2, 3, 4, 5, 6]

compose(double, add_ten, square)(3) = 38
  Step by step: square(3)=9, add_ten(9)=19, double(19)=38


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.6  BONUS: sorted, min, max, any, all with key parameter
# ═══════════════════════════════════════════════════════════════════════════════

students = [
    {"name": "Alice",  "gpa": 3.9, "age": 22},
    {"name": "Bob",    "gpa": 3.5, "age": 25},
    {"name": "Carol",  "gpa": 3.9, "age": 21},
    {"name": "Dave",   "gpa": 3.2, "age": 23},
]

# sorted with key
by_gpa    = sorted(students, key=lambda s: s["gpa"], reverse=True)
by_name   = sorted(students, key=lambda s: s["name"])
by_gpa_age = sorted(students, key=lambda s: (-s["gpa"], s["age"]))  # multi-key sort

print("Sorted by GPA desc     :", [s["name"] for s in by_gpa])
print("Sorted by name         :", [s["name"] for s in by_name])
print("Sorted by GPA, then age:", [s["name"] for s in by_gpa_age])

# min/max with key
youngest = min(students, key=lambda s: s["age"])
top_gpa  = max(students, key=lambda s: s["gpa"])
print(f"\nYoungest: {youngest['name']} (age {youngest['age']})")
print(f"Top GPA : {top_gpa['name']} (GPA {top_gpa['gpa']})")

# any / all
scores = [88, 92, 45, 76, 95]
print(f"\nany score > 90?    {any(s > 90 for s in scores)}")
print(f"all scores >= 50?  {all(s >= 50 for s in scores)}")
print(f"all scores >= 40?  {all(s >= 40 for s in scores)}")

# any/all short-circuit - they stop as soon as result is known
# Useful for large datasets

Sorted by GPA desc     : ['Alice', 'Carol', 'Bob', 'Dave']
Sorted by name         : ['Alice', 'Bob', 'Carol', 'Dave']
Sorted by GPA, then age: ['Carol', 'Alice', 'Bob', 'Dave']

Youngest: Carol (age 21)
Top GPA : Alice (GPA 3.9)

any score > 90?    True
all scores >= 50?  False
all scores >= 40?  True


---
# PART 4 - *args AND **kwargs

## Theory

Python functions accept 5 types of parameters:

| Type | Syntax | What it does |
|------|--------|--------------|
| Positional | `def f(a, b)` | Fixed position |
| Default | `def f(a, b=10)` | Optional with default |
| `*args` | `def f(*args)` | Any number of positional args → tuple |
| Keyword-only | `def f(*, key)` | Must be passed as keyword |
| `**kwargs` | `def f(**kwargs)` | Any number of keyword args → dict |

**Order rule:** `def f(pos, default, *args, kw_only, **kwargs)`

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.1  *args - variable positional arguments
# ═══════════════════════════════════════════════════════════════════════════════

def total(*args):
    """
    Accept any number of positional arguments.
    args is a TUPLE inside the function.
    """
    print(f"  args = {args}  | type: {type(args)}")
    return sum(args)


print(total(1, 2, 3))             # 3 args
print(total(10, 20, 30, 40, 50))  # 5 args
print(total())                    # 0 args → sum of empty tuple = 0

# *args after positional params
def greet(greeting, *names):
    """First arg is fixed, rest are collected into *names."""
    for name in names:
        print(f"  {greeting}, {name}!")

print()
greet("Hello", "Alice", "Bob", "Carol")

# Unpacking a list into *args with *
nums = [1, 2, 3, 4, 5]
print(f"\ntotal(*nums) = {total(*nums)}")   # * unpacks the list

  args = (1, 2, 3)  | type: <class 'tuple'>
6
  args = (10, 20, 30, 40, 50)  | type: <class 'tuple'>
150
  args = ()  | type: <class 'tuple'>
0

  Hello, Alice!
  Hello, Bob!
  Hello, Carol!
  args = (1, 2, 3, 4, 5)  | type: <class 'tuple'>

total(*nums) = 15


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.2  **kwargs - variable keyword arguments
# ═══════════════════════════════════════════════════════════════════════════════

def display_info(**kwargs):
    """
    Accept any number of keyword arguments.
    kwargs is a DICT inside the function.
    """
    print(f"  kwargs = {kwargs}  | type: {type(kwargs)}")
    for key, value in kwargs.items():
        print(f"    {key}: {value}")


display_info(name="Alice", age=30, role="Engineer", city="Pune")

# Unpack a dict into **kwargs with **
config = {"host": "localhost", "port": 5432, "db": "mydb"}
print()
display_info(**config)          # ** unpacks the dict

# Real-world: building flexible API request function
def api_request(endpoint: str, method: str = "GET", **params):
    """
    Simulate an API call with optional query params.
    **params captures any extra keyword arguments.
    """
    query = "&".join(f"{k}={v}" for k, v in params.items())
    url   = f"{endpoint}?{query}" if query else endpoint
    print(f"  [{method}] {url}")

print()
api_request("/api/users")
api_request("/api/products", page=1, limit=20, sort="price")
api_request("/api/orders", method="POST", user_id=42, status="pending")

  kwargs = {'name': 'Alice', 'age': 30, 'role': 'Engineer', 'city': 'Pune'}  | type: <class 'dict'>
    name: Alice
    age: 30
    role: Engineer
    city: Pune

  kwargs = {'host': 'localhost', 'port': 5432, 'db': 'mydb'}  | type: <class 'dict'>
    host: localhost
    port: 5432
    db: mydb

  [GET] /api/users
  [GET] /api/products?page=1&limit=20&sort=price
  [POST] /api/orders?user_id=42&status=pending


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.3  ALL PARAMETER TYPES COMBINED + KEYWORD-ONLY
# ═══════════════════════════════════════════════════════════════════════════════

def full_example(
    pos1,              # Positional only (required)
    pos2,              # Positional only (required)
    default1=10,       # Optional with default
    *args,             # Any extra positional → tuple
    kw_only,           # Keyword-only (must pass as keyword=value)
    kw_default="hi",   # Keyword-only with default
    **kwargs           # Any extra keyword args → dict
):
    print(f"  pos1      = {pos1}")
    print(f"  pos2      = {pos2}")
    print(f"  default1  = {default1}")
    print(f"  args      = {args}")
    print(f"  kw_only   = {kw_only}")
    print(f"  kw_default= {kw_default}")
    print(f"  kwargs    = {kwargs}")


full_example(
    "A", "B",          # pos1, pos2
    99,                # default1
    "X", "Y", "Z",     # extra positional → args
    kw_only="must",    # keyword-only (required)
    kw_default="bye",  # keyword-only with override
    extra1=1,          # → kwargs
    extra2=2           # → kwargs
)

# ── Positional-only params (Python 3.8+, before /) ────────────────────────────
def pos_only(x, y, /, z):    # x, y must be positional; z can be keyword
    return x + y + z

print(f"\npos_only(1, 2, z=3) = {pos_only(1, 2, z=3)}")
print(f"pos_only(1, 2, 3)   = {pos_only(1, 2, 3)}")
try:
    pos_only(x=1, y=2, z=3)   # Will fail - x and y are positional-only
except TypeError as e:
    print(f"Error: {e}")

  pos1      = A
  pos2      = B
  default1  = 99
  args      = ('X', 'Y', 'Z')
  kw_only   = must
  kw_default= bye
  kwargs    = {'extra1': 1, 'extra2': 2}

pos_only(1, 2, z=3) = 6
pos_only(1, 2, 3)   = 6
Error: pos_only() got some positional-only arguments passed as keyword arguments: 'x, y'


In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.4  FORWARDING args/kwargs - decorator pattern
# ═══════════════════════════════════════════════════════════════════════════════

def logger(func):
    """
    Simple decorator that logs function calls.
    *args, **kwargs allow it to wrap ANY function signature.
    """
    def wrapper(*args, **kwargs):
        print(f"  [LOG] Calling {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)   # Forward all arguments
        print(f"  [LOG] {func.__name__} returned {result}")
        return result
    return wrapper


@logger
def add(a, b):
    return a + b

@logger
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"


add(3, 4)
print()
greet("Alice")
print()
greet("Bob", greeting="Namaste")

  [LOG] Calling add(args=(3, 4), kwargs={})
  [LOG] add returned 7

  [LOG] Calling greet(args=('Alice',), kwargs={})
  [LOG] greet returned Hello, Alice!

  [LOG] Calling greet(args=('Bob',), kwargs={'greeting': 'Namaste'})
  [LOG] greet returned Namaste, Bob!


'Namaste, Bob!'

---
# PART 5 - SHALLOW vs DEEP COPY

## Theory

This is one of the most common bugs in Python - especially when working with mutable objects.

| Operation | What happens | When to use |
|-----------|-------------|-------------|
| **Assignment** `b = a` | Both point to SAME object | Never for mutable data you want independent |
| **Shallow copy** `b = a.copy()` | New outer object, but **nested objects are shared** | Flat structures (no nesting) |
| **Deep copy** `copy.deepcopy(a)` | New object at every level - fully independent | Nested structures |

```
SHALLOW COPY:             DEEP COPY:
a → [●, ●, ●]            a → [●, ●, ●]
b → [●, ●, ●]            b → [●, ●, ●]
      ↑↑↑ same              ↑↑↑ different objects
      inner objects
```

In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.1  ASSIGNMENT - NOT a copy
# ═══════════════════════════════════════════════════════════════════════════════

a = [1, 2, [3, 4]]
b = a                   # b is just another name for the SAME list

print(f"Before: a={a}, b={b}")
print(f"Same object? {a is b} | id(a)={id(a)}, id(b)={id(b)}")

b.append(99)
print(f"After b.append(99): a={a}")   # a is also changed!

Before: a=[1, 2, [3, 4]], b=[1, 2, [3, 4]]
Same object? True | id(a)=2501551179584, id(b)=2501551179584
After b.append(99): a=[1, 2, [3, 4], 99]


In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.2  SHALLOW COPY - new outer object, shared inner
# ═══════════════════════════════════════════════════════════════════════════════

import copy

original = [1, 2, [10, 20], [30, 40]]

# Three ways to shallow copy a list
sc1 = original.copy()         # list method
sc2 = original[:]             # slicing
sc3 = list(original)          # list() constructor
sc4 = copy.copy(original)     # copy.copy()

print(f"original is sc1? {original is sc1}")     # False - different objects
print(f"original[2] is sc1[2]? {original[2] is sc1[2]}")  # True - SAME inner list!

# Mutate outer - independent
sc1.append(999)
print(f"\nAfter sc1.append(999):")
print(f"  original = {original}")
print(f"  sc1      = {sc1}")

# Mutate inner - BOTH affected!
sc1[2].append(99)             # sc1[2] and original[2] are the SAME list
print(f"\nAfter sc1[2].append(99):")
print(f"  original = {original}")
print(f"  sc1      = {sc1}")
print("  ↑ BOTH changed because inner list is shared!")

original is sc1? False
original[2] is sc1[2]? True

After sc1.append(999):
  original = [1, 2, [10, 20], [30, 40]]
  sc1      = [1, 2, [10, 20], [30, 40], 999]

After sc1[2].append(99):
  original = [1, 2, [10, 20, 99], [30, 40]]
  sc1      = [1, 2, [10, 20, 99], [30, 40], 999]
  ↑ BOTH changed because inner list is shared!


In [20]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.3  DEEP COPY - fully independent at every level
# ═══════════════════════════════════════════════════════════════════════════════

original = [1, 2, [10, 20], {"key": "value"}]

deep = copy.deepcopy(original)

print(f"original is deep? {original is deep}")            # False
print(f"original[2] is deep[2]? {original[2] is deep[2]}") # False - independent!
print(f"original[3] is deep[3]? {original[3] is deep[3]}") # False - independent!

# Mutate inner - NO side effects
deep[2].append(99)
deep[3]["key"] = "CHANGED"

print(f"\nAfter modifying deep:")
print(f"  original = {original}")
print(f"  deep     = {deep}")
print("  ↑ original is untouched!")

# ── Summary table ──────────────────────────────────────────────────────────────
print("""
┌────────────────┬─────────────────────┬────────────────────┐
│ Operation      │ Outer object        │ Inner objects      │
├────────────────┼─────────────────────┼────────────────────┤
│ b = a          │ SAME                │ SAME               │
│ b = a.copy()   │ NEW (independent)   │ SAME (shared)      │
│ deepcopy(a)    │ NEW (independent)   │ NEW (independent)  │
└────────────────┴─────────────────────┴────────────────────┘
""")

# ── When deep copy is essential ───────────────────────────────────────────────
# Example: game board - you want to explore a move without affecting the original
board = [[".", ".", "."], [".", "X", "."], [".", ".", "."]]
test_board = copy.deepcopy(board)
test_board[0][0] = "O"          # Try a move
print(f"Original board[0][0]: {board[0][0]}")       # Still '.'
print(f"Test board[0][0]    : {test_board[0][0]}")  # 'O'

original is deep? False
original[2] is deep[2]? False
original[3] is deep[3]? False

After modifying deep:
  original = [1, 2, [10, 20], {'key': 'value'}]
  deep     = [1, 2, [10, 20, 99], {'key': 'CHANGED'}]
  ↑ original is untouched!

┌────────────────┬─────────────────────┬────────────────────┐
│ Operation      │ Outer object        │ Inner objects      │
├────────────────┼─────────────────────┼────────────────────┤
│ b = a          │ SAME                │ SAME               │
│ b = a.copy()   │ NEW (independent)   │ SAME (shared)      │
│ deepcopy(a)    │ NEW (independent)   │ NEW (independent)  │
└────────────────┴─────────────────────┴────────────────────┘

Original board[0][0]: .
Test board[0][0]    : O


---
# Part 1 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Iterators | Implement `__iter__` + `__next__`. `for` loop uses this protocol. |
| Generators | `yield` pauses execution. Memory-efficient for large/infinite data. |
| `yield from` | Delegate to sub-generator cleanly. |
| `zip` | Pair iterables. Stops at shortest. Use `zip_longest` otherwise. |
| `enumerate` | Always use instead of `range(len(x))`. |
| `map/filter` | Return lazy iterators. Often replaced by comprehensions. |
| `reduce` | Fold to single value. Import from `functools`. |
| `*args` | Variable positional → tuple inside function. |
| `**kwargs` | Variable keyword → dict inside function. |
| Shallow copy | New outer, shared inner. Use for flat data. |
| Deep copy | Fully independent. Always use with nested mutable data. |

---
**→ Continue to Part 2 (OOP) notebook**
